# Lead assignment (notebook version)

Notebook equivalent of `assign_leads.py`: reads the new leads and the active sellers,
assigns each lead to a seller based on **language** and **content of interest**, and
materializes the result in `lead_assignments` (ClickHouse, `DB_NAME` database).

```
leads_clean  +  sales_people  ->  lead_assignments
```

All the logic lives in the `utils/` package; this notebook only orchestrates the steps.

## 1. Imports

`assign_leads.py` uses package-relative imports (`from .utils.db import ...`). Those only work
inside a module that belongs to a package, and a notebook is not one, so `from .utils ...` raises
`ImportError: attempted relative import with no known parent package`.

So we put this folder on `sys.path` and import `utils` from it. The working directory is not always
this folder: Jupyter uses it, but a scheduler running papermill (like Intelek) may start from the repo
root. The next cell looks for `utils/` from the working directory, `python_ipynb/` and
`compute/python_ipynb/`. Inside `utils/`, the
modules still use relative imports between themselves (`from .candidates import ...`).

In [11]:
import sys
from pathlib import Path

import pandas as pd

## A notebook does not know where its own file is, and the working directory depends on who runs it:
## Jupyter uses the notebook's folder, a scheduler (papermill) may use the repo root.
## Find the folder that contains utils/ from any of them.
#CANDIDATES = [Path.cwd(), Path.cwd() / "python_ipynb", Path.cwd() / "compute" / "python_ipynb"]
#NOTEBOOK_DIR = next((d for d in CANDIDATES if (d / "utils" / "db.py").is_file()), None)
#if NOTEBOOK_DIR is None:
#    raise ModuleNotFoundError(f"utils/ not found from {Path.cwd()}: run from the repo root, "
#                              "compute/ or compute/python_ipynb/")
#if str(NOTEBOOK_DIR) not in sys.path:
#    sys.path.insert(0, str(NOTEBOOK_DIR))

from utils.db import get_client, read_table, table_exists, write_table
from utils.matching import assign_leads, current_load, normalize_sellers, summarize

## 2. Parameters

The next cell is tagged `parameters`: when Intelek runs the notebook with papermill, it
injects a cell right after it with the actual values (including the `creds` secret).

In [2]:
# Intelek injects `creds` (the secret object, a dict) in the first cell of the notebook.
# Keep it if it is already defined; otherwise (local runs) None -> read compute/.env
try:
    creds
except NameError:
    creds = None
LEADS_TABLE = "leads_clean"
SELLERS_TABLE = "sales_people"
OUTPUT_TABLE = "lead_assignments"

FULL_REFRESH = False   # True: ignore previous assignments and rebuild the table
DRY_RUN = False        # True: compute but do not write

In [3]:
client = get_client(creds)

## 3. Read inputs

In [4]:
leads = read_table(client, LEADS_TABLE)
sellers = normalize_sellers(read_table(client, SELLERS_TABLE))

print(f"{len(leads)} leads | {len(sellers)} active sellers")
sellers

118 leads | 8 active sellers


,seller_id,full_name,email,languages,specialties,max_open_leads,is_active
0,S01,Laura Martí,laura.marti@prenomics-demo.com,"frozenset({en, ca, es})","frozenset({data_governance, data_engineering})",15,True
1,S02,Marc Puig,marc.puig@prenomics-demo.com,"frozenset({ca, es})","frozenset({bi_analytics, cloud_platforms})",12,True
2,S03,Sophie Dubois,sophie.dubois@prenomics-demo.com,"frozenset({en, fr})","frozenset({genai, machine_learning})",12,True
3,S04,Jonas Weber,jonas.weber@prenomics-demo.com,"frozenset({en, de})","frozenset({cloud_platforms, data_engineering})",10,True
4,S05,Ana Costa,ana.costa@prenomics-demo.com,"frozenset({pt, en, es})","frozenset({data_governance, bi_analytics})",10,True
5,S06,Pablo Ruiz,pablo.ruiz@prenomics-demo.com,"frozenset({en, es})","frozenset({genai, machine_learning})",15,True
6,S07,Emma Clarke,emma.clarke@prenomics-demo.com,frozenset({en}),"frozenset({genai, bi_analytics, data_engineeri...",14,True
7,S09,Thomas Lefèvre,thomas.lefevre@prenomics-demo.com,"frozenset({en, de, fr})","frozenset({cloud_platforms, bi_analytics, data...",10,True


## 4. Keep only the new leads

A lead is *new* if it is not in the output table yet. With `FULL_REFRESH = True` every lead is reassigned.

In [5]:
existing = None
if not FULL_REFRESH and table_exists(client, OUTPUT_TABLE):
    existing = read_table(client, OUTPUT_TABLE)
    leads = leads[~leads["lead_id"].isin(existing["lead_id"])]

load = current_load(existing)   # {seller_id: leads already assigned}
print(f"{len(leads)} new leads | {0 if existing is None else len(existing)} previous assignments")
load

0 new leads | 118 previous assignments


{'S01': 15,
 'S06': 15,
 'S07': 13,
 'S02': 12,
 'S03': 11,
 'S05': 10,
 'S04': 10,
 'S09': 10}

## 5. Assign

Rules, lead by lead in order of arrival:
1. Prefer sellers who speak the lead's language **and** are specialized in its content (`full_match`).
2. Otherwise, any seller speaking the language (`language_only`).
3. Within each tier, the seller with the lowest load / capacity ratio wins. If nobody is available the lead stays `unassigned`.

In [6]:
assignments = assign_leads(leads, sellers, load=load)
summarize(assignments)

match_type,full_match,language_only,unassigned
seller_id,,,


In [7]:
# Leads that did not get a perfect match, and why
assignments[assignments["match_type"] != "full_match"]

,lead_id,seller_id,match_type,reason,preferred_language,content_of_interest,assigned_at,run_id


## 6. Materialize

In [8]:
if assignments.empty:
    print("Nothing to assign.")
elif DRY_RUN:
    print("Dry run: nothing written.")
else:
    write_table(assignments, client, OUTPUT_TABLE, if_exists="replace" if FULL_REFRESH else "append")
    print(f"{len(assignments)} rows written to {OUTPUT_TABLE}")

Nothing to assign.


## 7. Check the output table

In [9]:
client.query_df(f"""
    SELECT coalesce(seller_id, '-') AS seller_id, match_type, count(*) AS n_leads
    FROM {OUTPUT_TABLE}
    GROUP BY 1, 2
    ORDER BY 1, 2
""")

,seller_id,match_type,n_leads
0,-,unassigned,22
1,S01,full_match,10
2,S01,language_only,5
3,S02,full_match,12
4,S03,full_match,9
5,S03,language_only,2
6,S04,full_match,5
7,S04,language_only,5
8,S05,full_match,5
9,S05,language_only,5
